# ETL pipeline outline
1. Write a script to Take the file and send it to another location As it's being sent in real time in batches 
2. store these data into hdfs and apply some transformations 
3. build star schema 
4. load the data into snowflake DWH 
5. put all of this into airflow 

In [1]:
import os
print(os.getcwd())    # check if i'm inside docker

/home/jovyan/work


In [2]:
# Hadoop:
# hadoop-namenode
# hadoop-datanode1
# hadoop-datanode2

# YARN:
# resourcemanager
# hadoop-nodemanager1
# hadoop-nodemanager2

# Spark:
# spark-jupyter

### 1. Take the file and send it to another location As it's being sent in real time in batches

In [3]:
# tell Hadoop to treat your Spark session as the root user or the hdfs user. 
# You can do this by setting an environment variable before you initialize your SparkSession.
os.environ["HADOOP_USER_NAME"] = "root"

In [4]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import  *


In [5]:
spark = SparkSession.builder \
    .appName('starSchemaTrasnformations') \
    .master('yarn') \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-namenode:9000") \
    .config("spark.hadoop.yarn.resourcemanager.hostname", "resourcemanager") \
    .config("spark.hadoop.yarn.resourcemanager.address", "resourcemanager:8032") \
    .config("spark.hadoop.yarn.resourcemanager.scheduler.address", "resourcemanager:8030") \
    .config("spark.driver.host", "172.30.1.13") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.executor.memory", "512m") \
    .config("spark.yarn.am.memory", "512m") \
    .config("spark.jars.packages", "net.snowflake:spark-snowflake_2.12:2.12.0-spark_3.3,net.snowflake:snowflake-jdbc:3.13.22") \
    .getOrCreate()
 

In [6]:
schema = StructType([
    StructField("carat", DoubleType(), True),
    StructField("cut", StringType(), True),
    StructField("color", StringType(), True),
    StructField("clarity", StringType(), True),
    StructField("depth", DoubleType(), True),
    StructField("table", DoubleType(), True),
    StructField("price", IntegerType(), True),
    StructField("x", DoubleType(), True),
    StructField("y", DoubleType(), True),
    StructField("z", DoubleType(), True)
])

#### Batch reading

In [7]:
# recall
# Batch (read): Grabs all files currently in /data and treats them as one large table.
# Streaming (readStream): Watches the directory. It will treat the 3 existing files as the "first batch" and then wait for any new files to be dropped into the folder.

In [8]:
df = spark.read \
    .option("header", True) \
    .schema(schema) \
    .csv("hdfs://hadoop-namenode:9000/data")

df.show(15) 
print(f"Total rows: {df.count()}")

+-----+---------+-----+-------+-----+-----+-----+----+----+----+
|carat|      cut|color|clarity|depth|table|price|   x|   y|   z|
+-----+---------+-----+-------+-----+-----+-----+----+----+----+
| 0.23|    Ideal|    E|    SI2| 61.5| 55.0|  326|3.95|3.98|2.43|
| 0.21|  Premium|    E|    SI1| 59.8| 61.0|  326|3.89|3.84|2.31|
| 0.23|     Good|    E|    VS1| 56.9| 65.0|  327|4.05|4.07|2.31|
| 0.29|  Premium|    I|    VS2| 62.4| 58.0|  334| 4.2|4.23|2.63|
| 0.31|     Good|    J|    SI2| 63.3| 58.0|  335|4.34|4.35|2.75|
| 0.24|Very Good|    J|   VVS2| 62.8| 57.0|  336|3.94|3.96|2.48|
| 0.24|Very Good|    I|   VVS1| 62.3| 57.0|  336|3.95|3.98|2.47|
| 0.26|Very Good|    H|    SI1| 61.9| 55.0|  337|4.07|4.11|2.53|
| 0.22|     Fair|    E|    VS2| 65.1| 61.0|  337|3.87|3.78|2.49|
| 0.23|Very Good|    H|    VS1| 59.4| 61.0|  338| 4.0|4.05|2.39|
|  0.3|     Good|    J|    SI1| 64.0| 55.0|  339|4.25|4.28|2.73|
| 0.23|    Ideal|    J|    VS1| 62.8| 56.0|  340|3.93| 3.9|2.46|
| 0.22|  Premium|    F|  

In [9]:
# creating a streaming dataframe (lazy - microbatches - schema is required)
# df = spark.readStream \
#     .option("header", True) \
#     .schema(schema) \
#     .csv("hdfs://hadoop-namenode:9000/data")

In [10]:
df.printSchema()

root
 |-- carat: double (nullable = true)
 |-- cut: string (nullable = true)
 |-- color: string (nullable = true)
 |-- clarity: string (nullable = true)
 |-- depth: double (nullable = true)
 |-- table: double (nullable = true)
 |-- price: integer (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- z: double (nullable = true)



### 2. Transformation 

In [11]:
# carat,cut,color,clarity,depth,table,price,x,y,z

In [12]:
transformed_df = df \
    .withColumn("volume_mm3", col("x") * col("y") * col("z")) \
    .withColumn("processed_timestamp", current_timestamp()) \
    .filter(col("price") > 0)

In [13]:
rename_map = {"x": "length_mm", "y": "width_mm", "z": "depth_mm"}
for old, new in rename_map.items():
    transformed_df = transformed_df.withColumnRenamed(old, new)


In [14]:
transformed_df.show(5)

+-----+-------+-----+-------+-----+-----+-----+---------+--------+--------+------------------+--------------------+
|carat|    cut|color|clarity|depth|table|price|length_mm|width_mm|depth_mm|        volume_mm3| processed_timestamp|
+-----+-------+-----+-------+-----+-----+-----+---------+--------+--------+------------------+--------------------+
| 0.23|  Ideal|    E|    SI2| 61.5| 55.0|  326|     3.95|    3.98|    2.43|          38.20203|2026-05-08 22:02:...|
| 0.21|Premium|    E|    SI1| 59.8| 61.0|  326|     3.89|    3.84|    2.31|         34.505856|2026-05-08 22:02:...|
| 0.23|   Good|    E|    VS1| 56.9| 65.0|  327|     4.05|    4.07|    2.31|         38.076885|2026-05-08 22:02:...|
| 0.29|Premium|    I|    VS2| 62.4| 58.0|  334|      4.2|    4.23|    2.63|          46.72458|2026-05-08 22:02:...|
| 0.31|   Good|    J|    SI2| 63.3| 58.0|  335|     4.34|    4.35|    2.75|51.917249999999996|2026-05-08 22:02:...|
+-----+-------+-----+-------+-----+-----+-----+---------+--------+------

In [15]:
# preventing data entry errors
transformed_df = transformed_df.withColumn("depth_pct_calc", round(100.0 * col("depth_mm") / ((col("length_mm") + col("width_mm"))/2.0), 2))
transformed_df = transformed_df.withColumn("depth_diff", round(col("depth_pct_calc") - col("depth"), 3))

In [16]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

cut_map = {"Fair":1, "Good":2, "Very Good":3, "Premium":4, "Ideal":5}
transformed_df.select("cut").distinct().withColumn("cut_id", row_number().over(Window.orderBy("cut")))

DataFrame[cut: string, cut_id: int]

In [17]:
transformed_df = transformed_df.withColumn("price_per_carat", round(col("price")/col("carat"),2))

In [18]:
# query = df.writeStream \
#     .outputMode("append") \
#     .format("parquet") \
#     .option("path", "hdfs://hadoop-namenode:9000/refined_diamonds/") \
#     .option("checkpointLocation", "hdfs://hadoop-namenode:9000/checkpoints/") \
#     .start()

In [19]:
# Pass the schema manually when reading!
# df = spark.read \
#     .schema(schema) \
#     .parquet("hdfs://hadoop-namenode:9000/data/")

# # temporary SQL view
# df.createOrReplaceTempView("historical_diamonds")

### Querying the Final Output

In [20]:
pd_df = transformed_df.toPandas()
pd_df.head()

,carat,cut,color,clarity,depth,table,price,length_mm,width_mm,depth_mm,volume_mm3,processed_timestamp,depth_pct_calc,depth_diff,price_per_carat
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43,38.202030,2026-05-08 22:02:59.310407,61.29,-0.21,1417.39
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31,34.505856,2026-05-08 22:02:59.310407,59.77,-0.03,1552.38
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31,38.076885,2026-05-08 22:02:59.310407,56.90,0.00,1421.74
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63,46.724580,2026-05-08 22:02:59.310407,62.40,0.00,1151.72
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75,51.917250,2026-05-08 22:02:59.310407,63.29,-0.01,1080.65


In [21]:
desc = transformed_df.describe().toPandas()
desc = desc.set_index("summary").T.reset_index().rename(columns={"index":"cut"})
desc

summary,cut,count,mean,stddev,min,max
0,carat,53943,0.7979346717832034,0.47399855309645916,0.2,5.01
1,cut,53943,None,None,Fair,Very Good
2,color,53943,None,None,D,J
3,clarity,53943,None,None,I1,VVS2
4,depth,53943,61.749322432938364,1.4326262660399924,43.0,79.0
5,table,53943,57.45725117253398,2.2345489532743477,43.0,95.0
6,price,53943,3932.734293606214,3989.338446824254,326,18823
7,length_mm,53943,5.731158074263546,1.1217295857901184,0.0,10.74
8,width_mm,53943,5.734526444580368,1.1421029192821053,0.0,58.9
9,depth_mm,53943,3.5387295849322755,0.7056794713896594,0.0,31.8


In [22]:
transformed_df.selectExpr("cut","color", "clarity").show(5)

+-------+-----+-------+
|    cut|color|clarity|
+-------+-----+-------+
|  Ideal|    E|    SI2|
|Premium|    E|    SI1|
|   Good|    E|    VS1|
|Premium|    I|    VS2|
|   Good|    J|    SI2|
+-------+-----+-------+
only showing top 5 rows



In [23]:
transformed_df.groupBy("cut").agg(round(avg("carat"), 3).alias("Avg Carat")).show()

+---------+---------+
|      cut|Avg Carat|
+---------+---------+
|  Premium|    0.892|
|    Ideal|    0.703|
|     Good|    0.849|
|     Fair|    1.046|
|Very Good|    0.806|
+---------+---------+



In [24]:
transformed_df.where("price IS NOT NULL").orderBy(("price"))\
.select(col("carat"), col("cut"), col("color"),col("price"),).show(6)

+-----+---------+-----+-----+
|carat|      cut|color|price|
+-----+---------+-----+-----+
| 0.23|    Ideal|    E|  326|
| 0.21|  Premium|    E|  326|
| 0.23|     Good|    E|  327|
| 0.29|  Premium|    I|  334|
| 0.31|     Good|    J|  335|
| 0.24|Very Good|    J|  336|
+-----+---------+-----+-----+
only showing top 6 rows



In [25]:
from pyspark.sql.functions import corr
transformed_df.select(corr("carat", "price")).show()

+------------------+
|corr(carat, price)|
+------------------+
|0.9215912778016075|
+------------------+



In [26]:
transformed_df.printSchema()

root
 |-- carat: double (nullable = true)
 |-- cut: string (nullable = true)
 |-- color: string (nullable = true)
 |-- clarity: string (nullable = true)
 |-- depth: double (nullable = true)
 |-- table: double (nullable = true)
 |-- price: integer (nullable = true)
 |-- length_mm: double (nullable = true)
 |-- width_mm: double (nullable = true)
 |-- depth_mm: double (nullable = true)
 |-- volume_mm3: double (nullable = true)
 |-- processed_timestamp: timestamp (nullable = false)
 |-- depth_pct_calc: double (nullable = true)
 |-- depth_diff: double (nullable = true)
 |-- price_per_carat: double (nullable = true)



### Star Schema

In [27]:
# HDFS UI → http://localhost:9870
# YARN UI → http://localhost:8088

```
fact_diamonds (
    id PK, 
    cut_id INT  FK,
    color_id INT  FK,
    clarity_id INT  FK,
    date_id  INT  FK,

    carat FLOAT,
    depth FLOAT,
    table FLOAT,

    length_mm (Double),
    width_mm (Double),
    depth_mm (Double),
    price INT, 

    volume_mm3 (Double),
    depth_pct_calc (Double),
    depth_diff (Double),
    price_per_carat (Double),
    processed_timestamp
)

dim_cut (
    cut_id INT PRIMARY K    cut_name STRING,
    
)

dim_color (
    color_id INT PRIMARY KEY,
    color_name STRING
)

dim_clarity (
    clarity_id INT PRIMARY KEY,
    clarity_name STRING
)

Dim_Date (
  date_id (Primary Key, INT),
  quarter  INT,
  day INT,
 ``` month INT,
  year INT,
  full_date DATE
)


In [28]:
# carat	cut	color	clarity	depth	table	price	length_mm	width_mm	depth_mm	volume_mm3	processed_timestamp	depth_pct_calc	depth_diff	price_per_carat

In [29]:
BASE_PATH = "hdfs://hadoop-namenode:9000/user/root/output/" 

In [30]:
# DimCut
dim_cut = (transformed_df.select("cut").distinct()
           .na.fill({"cut": "UNKNOWN"})
           .withColumn("cut_id", row_number().over(Window.orderBy("cut")))
           .select("cut_id", col("cut").alias("cut_name"))
           .write.mode("overwrite").parquet(f"{BASE_PATH}dim_cut")
)

In [31]:
# DimColor
dim_color = (transformed_df.select("color").distinct()
             .na.fill({"color": "UNKNOWN"})
             .withColumn("color_id", row_number().over(Window.orderBy("color")))
             .select("color_id", col("color").alias("color_name"))
             .write.mode("overwrite").parquet(f"{BASE_PATH}dim_color")
)

In [32]:
# DimClarity
dim_clarity = (transformed_df.select("clarity").distinct()
               .na.fill({"clarity": "UNKNOWN"})
               .withColumn("clarity_id", row_number().over(Window.orderBy("clarity")))
               .select("clarity_id", col("clarity").alias("clarity_name"))
               .write.mode("overwrite").parquet(f"{BASE_PATH}dim_clarity")
)

In [33]:
# DimDate 
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import to_date, year, month, dayofmonth, quarter
dim_date = (transformed_df.select(to_date(col("processed_timestamp")).alias("full_date"))
            .distinct()
            .withColumn("date_id", row_number().over(Window.orderBy("full_date")))
            .withColumn("year", year(col("full_date")))
            .withColumn("month", month(col("full_date")))
            .withColumn("day", dayofmonth(col("full_date")))
            .withColumn("quarter", quarter(col("full_date")))
            .select("date_id","quarter","day","month","year","full_date")
            .write.mode("overwrite").parquet(f"{BASE_PATH}dim_date")
)

In [34]:
dim_cut = spark.read.parquet(f"{BASE_PATH}dim_cut")
dim_color = spark.read.parquet(f"{BASE_PATH}dim_color")
dim_clarity = spark.read.parquet(f"{BASE_PATH}dim_clarity")
dim_date = spark.read.parquet(f"{BASE_PATH}dim_date")

# join to get FK ids
fact = (transformed_df
        .join((dim_cut), transformed_df.cut == dim_cut.cut_name, "left")
        .join((dim_color), transformed_df.color == dim_color.color_name, "left")
        .join((dim_clarity), transformed_df.clarity == dim_clarity.clarity_name, "left")
        .join((dim_date), to_date(transformed_df.processed_timestamp) == dim_date.full_date, "left")
       )

# Select and reorder columns to match your schema
fact = fact.select(
    col("cut_id"),
    col("color_id"),
    col("clarity_id"),
    col("date_id"),
    col("carat"),
    col("depth"),
    col("table"),
    col("length_mm"),
    col("width_mm"),
    col("depth_mm"),
    col("price"),
    col("volume_mm3"),
    col("depth_pct_calc"),
    col("depth_diff"),
    col("price_per_carat"),
    col("processed_timestamp")
)

In [37]:
fact.write.mode("overwrite").parquet(f"{BASE_PATH}fact_diamond")
print("Star Schema successfully generated and saved to HDFS!")

Star Schema successfully generated and saved to HDFS!


In [38]:
transformed_df.write \
    .mode("overwrite") \
    .parquet("hdfs://hadoop-namenode:9000/user/root/output/")